<a href="https://colab.research.google.com/github/charlien12/ML-Projects/blob/main/Copy_of_TextSummarizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer,TrainingArguments
import numpy as np
import pandas as pd

In [ ]:
train_df=pd.read_csv('sample_data/samsum-train.csv')
val_df=pd.read_csv('sample_data/samsum-validation.csv')

In [ ]:
train_df.fillna(train_df.mode().iloc[0], inplace=True)

# Data Preprocessing

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"<.*?>", "", text)          # remove HTML tags
    #text = re.sub(r"[^a-zA-Z0-9\s]", "", text) # remove special characters
    text = re.sub(r"\s+", " ", text)  #remove extra spaces
    text=re.sub(r"\r\n"," ",text)         # remove extra lines
    return text.strip().lower()

In [ ]:
train_df['dialogue']=train_df['dialogue'].apply(clean_text)
train_df['summary']=train_df['summary'].apply(clean_text)

val_df['dialogue']=val_df['dialogue'].apply(clean_text)
val_df['summary']=val_df['summary'].apply(clean_text)

In [ ]:
train_df.dialogue[0]

"amanda: i baked cookies. do you want some? jerry: sure! amanda: i'll bring you tomorrow :-)"

# Tokenization

In [ ]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [ ]:
def tokenize(data):
  inputs=tokenizer(data['dialogue'],padding="max_length",truncation=True,max_length=512)
  targets=tokenizer(data['summary'],padding="max_length",truncation=True,max_length=150)
  inputs["labels"]=targets["input_ids"]
  return inputs

In [ ]:
train_dataset=train_df.apply(tokenize,axis=1).tolist()
val_dataset=val_df.apply(tokenize,axis=1).tolist()

# Model

In [ ]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
if torch.cuda.is_available():
  device=torch.device("cuda")
elif torch.backends.mps.is_available():
  device=torch.device("mps")
else:
  device=torch.device("cpu")

print(device)

cuda


In [ ]:
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
training_args=TrainingArguments(output_dir="./results",
                                num_train_epochs=5,
                                weight_decay=0.01,
                                per_device_eval_batch_size=16,
                                per_device_train_batch_size=16,
                                eval_strategy="epoch",
                                save_strategy="epoch",
                                warmup_steps=500)

In [ ]:
trainer=Trainer(model=model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=val_dataset)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.506183,0.349812
2,0.382796,0.336719
3,0.362558,0.331480
4,0.355409,0.329505
5,0.352755,0.328543


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4605, training_loss=0.7080217309676345, metrics={'train_runtime': 3320.2926, 'train_samples_per_second': 22.185, 'train_steps_per_second': 1.387, 'total_flos': 9969277096427520.0, 'train_loss': 0.7080217309676345, 'epoch': 5.0})

# Save the Model

In [ ]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

### Test the core logic for summarization

In [ ]:
def summarize_dialogue(dialogue):
    dialogue = clean_text(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [ ]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai systems are becoming more capable due to advances in deep learning and access to large datasets. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
